# **Training Schema Implementation**

Author: Josh Chen + AI

### **1st training schema: class-balanced curriculum learning**

Summary: exposing models to different proportions of classes at different times.\
Phase 1: balanced distribution of classes (equal # of classes)\
Phase 2: mixed distribution between balanced and real distribution\
Phase 3: real distribution of classes\

In [6]:
from collections import defaultdict
import random

In [2]:
def build_class_index(df, label_col="localization"):
    class_to_indices = defaultdict(list)
    for idx, label in enumerate(df[label_col]):
        class_to_indices[label].append(idx)
    return class_to_indices

In [ ]:
class CurriculumSchedule:
    def __init__(self, total_epochs, phase1_frac=0.33, phase2_frac=0.33):
        self.total_epochs = total_epochs
        self.phase1_end = int(total_epochs * phase1_frac)
        self.phase2_end = int(total_epochs * (phase1_frac + phase2_frac))

    def get_phase(self, epoch):
        if epoch < self.phase1_end:
            return 1
        elif epoch < self.phase2_end:
            return 2
        else:
            return 3

In [ ]:
class ClassSampler:
    def __init__(self, class_to_indices, curriculum_schedule, batch_size, mix_alpha=0.5):
        """
        class_to_indices: dict mapping class → list of sample indices
        curriculum_schedule: object with get_phase(epoch)
        batch_size: number of samples per batch
        mix_alpha: fraction of balanced samples in Phase 2
        """
        self.class_to_indices = class_to_indices
        self.curriculum_schedule = curriculum_schedule
        self.classes = list(class_to_indices.keys())
        self.batch_size = batch_size
        self.mix_alpha = mix_alpha

        # Precompute class probabilities for realistic sampling
        counts = {cls: len(idxs) for cls, idxs in class_to_indices.items()}
        total = sum(counts.values())
        self.class_probs = {cls: counts[cls] / total for cls in counts}

    def sample(self, epoch):
        phase = self.curriculum_schedule.get_phase(epoch)

        if phase == 1:
            return self.sample_balanced()

        elif phase == 2:
            return self.sample_mixed()

        else:
            return self.sample_realistic()
    # -----------------------------
    # Phase 1: Balanced sampling
    # -----------------------------
    def sample_balanced(self):
        num_classes = len(self.classes)
        per_class = max(1, self.batch_size // num_classes)

        batch_indices = []
        for cls in self.classes:
            indices = self.class_to_indices[cls]
            # sample with replacement → oversampling rare classes
            chosen = random.choices(indices, k=per_class)
            batch_indices.extend(chosen)

        # Trim if overshooting
        if len(batch_indices) > self.batch_size:
            batch_indices = batch_indices[:self.batch_size]

        random.shuffle(batch_indices)
        return batch_indices


    # -----------------------------
    # Phase 2: Mixed sampling
    # -----------------------------
    def sample_mixed(self):
        # Number of balanced vs realistic samples
        k_balanced = int(self.batch_size * self.mix_alpha)
        k_realistic = self.batch_size - k_balanced

        # Balanced part
        balanced_indices = self.sample_balanced()[:k_balanced]

        # Realistic part
        realistic_indices = self.sample_realistic()[:k_realistic]

        batch_indices = balanced_indices + realistic_indices
        random.shuffle(batch_indices)
        return batch_indices


    # -----------------------------
    # Phase 3: Realistic sampling
    # -----------------------------
    def sample_realistic(self):
        batch_indices = []

        # Build probability list in correct order
        probs = [self.class_probs[c] for c in self.classes]

        for _ in range(self.batch_size):
            cls = random.choices(self.classes, weights=probs)[0]
            idx = random.choice(self.class_to_indices[cls])
            batch_indices.append(idx)

        return batch_indices
